In [8]:
import torch
import gc
from torchvision import transforms
import torch.nn as nn
from PIL import Image

torch.cuda.empty_cache()
gc.collect()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_activation=True, use_batchnorm=True, **kwargs):
        super().__init__()
        self.use_activation = use_activation
        self.cnn = nn.Conv2d(in_channels, out_channels, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels) if use_batchnorm else nn.Identity()
        self.ac = nn.LeakyReLU(0.2, inplace=True)
    
    def forward(self, x):
        cnn = self.cnn(x)
        bn = self.bn(cnn)
        out = self.ac(bn) if self.use_activation else bn
        return out

In [3]:
class UpsampleBlock(nn.Module):
    def __init__(self, in_channels, scale_factor):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * scale_factor**2, 2, 1, 1)
        self.ps = nn.PixelShuffle(scale_factor)
        self.ac = nn.PReLU(num_parameters=in_channels)
    
    def forward(self, x):
        return self.ac(self.ps(self.conv(x)))

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.b1 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1)
        self.b2 = ConvBlock(in_channels, in_channels, kernel_size=3, stride=1, padding=1, use_activation=False)
        
    def forward(self, x):
        b1 = self.b1(x)
        return b1 + self.b2(b1)

In [5]:
class Generator(nn.Module):
    def __init__(self, in_channels=3, num_channels=64, num_blocks=8):
        super().__init__()
        self.initial = ConvBlock(in_channels, num_channels, kernel_size=7, stride=1, padding=4, use_batchnorm=False)
        self.res = nn.Sequential(*[ResidualBlock(num_channels) for _ in range(num_blocks)])
        self.conv = ConvBlock(num_channels, num_channels, kernel_size=3, stride=1, padding=1, use_activation=False)
        self.up = nn.Sequential(UpsampleBlock(num_channels, scale_factor=2))
        self.final = nn.Conv2d(num_channels, in_channels, kernel_size=9, stride=1, padding=1)
    
    def forward(self, x):
        initial = self.initial(x)
        res = self.res(initial)
        conv = self.conv(res) + initial
        up = self.up(conv)
        out = self.final(up)
        return torch.sigmoid(out)

In [6]:
gen = Generator().to(device)
gen.load_state_dict(torch.load('checkpoint1_gen.pth', map_location=device))
gen.eval()

C:\Users\Himanshu Khandelwal\AppData\Local\Temp\ipykernel_5752\1901803313.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gen.load_state_dict(torch.load('checkpoint1_gen

Generator(
  (initial): ConvBlock(
    (cnn): Conv2d(3, 64, kernel_size=(7, 7), stride=(1, 1), padding=(4, 4))
    (bn): Identity()
    (ac): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (res): Sequential(
    (0): ResidualBlock(
      (b1): ConvBlock(
        (cnn): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (ac): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (b2): ConvBlock(
        (cnn): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (ac): LeakyReLU(negative_slope=0.2, inplace=True)
      )
    )
    (1): ResidualBlock(
      (b1): ConvBlock(
        (cnn): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (

In [18]:
low_res_img = Image.open('test/low/img1.jpg')
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((512, 512)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
low_res_tensor = preprocess(low_res_img).unsqueeze(0).to(device)

In [19]:
with torch.no_grad():
    high_res_tensor = gen(low_res_tensor)

high_res_image = transforms.ToPILImage()(high_res_tensor.squeeze(0).cpu())
high_res_image.save('test/high/img1.jpg')